# День 6 — Leakage, overfitting, Pipeline, CV

## Цель
Закрепить **Pipeline** (без leakage) и **cross-validation** — оценка модели стабильнее, чем один split.

## Данные и Pipeline

Titanic + Pipeline: scaler → LogisticRegression.
Все шаги предобработки **внутри** pipeline — fit только на train в каждом fold.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv('../../data/titanic.csv')
df = df.drop(columns=['Cabin'])
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X = df[features]
y = df['Survived']
print("Данные готовы!")

## Один train/test split

Accuracy на одном test — может «повезти» с разбиением. Смотрим baseline.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, solver="liblinear", random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")

## Cross-validation (cv=5)

Данные делят на **5 folds**. Модель обучается 5 раз, каждый раз другой fold — test.

- **Mean** — средняя accuracy
- **Std** — разброс (низкий std = стабильная модель)

⚠️ Imputer для Age лучше добавить в pipeline (day 4) — иначе пропуски не обработаны.

In [ ]:
scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')

print(f"Scores: {scores}")
print(f"Mean: {scores.mean():.2f}")
print(f"Std: {scores.std():.2f}")

## Выводы

- Pipeline объединяет preprocessing + модель — меньше риск leakage.
- Один test accuracy — снимок; CV даёт mean ± std.
- Низкий std → модель стабильна на разных разбиениях.

---

# Day 6 — Leakage, overfitting, Pipeline, CV

## Goal
Practice **Pipeline** (no leakage) and **cross-validation** — more stable than a single split.

## Data and Pipeline

Titanic + Pipeline: scaler → LogisticRegression.
All preprocessing **inside** the pipeline — fit on train only in each fold.

## Single train/test split

Accuracy on one test set can be lucky. We use it as a baseline.

## Cross-validation (cv=5)

Data split into **5 folds**. Model trains 5 times, each fold is test once.

- **Mean** — average accuracy
- **Std** — spread (low std = stable model)

⚠️ Add imputer for Age to the pipeline (day 4) — otherwise missing values are not handled.

## Conclusions

- Pipeline combines preprocessing + model — less leakage risk.
- Single test accuracy is a snapshot; CV gives mean ± std.
- Low std → model is stable across splits.